In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Filter parameters for backfill or standard run
dbutils.widgets.text("batch_id", "", "Batch ID")
dbutils.widgets.text("start_date", "", "Start Date (YYYY-MM-DD)")

BATCH_FILTER = dbutils.widgets.get("batch_id").strip()
START_DATE_FILTER = dbutils.widgets.get("start_date").strip()

CATALOG = "workspace"
DB = f"{CATALOG}.complaints"
BRONZE = f"{DB}.bronze_complaints"
SILVER = f"{DB}.silver_complaints"
QUARANTINE = f"{DB}.silver_quarantine"
LOGS = f"{DB}.pipeline_execution_logs"
WATERMARK = f"{DB}.pipeline_watermark"

BATCH_ID = BATCH_FILTER if BATCH_FILTER else f"batch_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"

# Parse timestamps safely using F.lit to avoid unresolved column errors
def parse_flexible_timestamp(col_name):
    return F.coalesce(
        F.try_to_timestamp(F.col(col_name)),
        F.try_to_timestamp(F.col(col_name), F.lit("yyyy-MM-dd")),
        F.try_to_timestamp(F.col(col_name), F.lit("yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")),
        F.try_to_timestamp(F.col(col_name), F.lit("yyyy-MM-dd'T'HH:mm:ssX"))
    )

# Get last processed watermark date
def get_watermark():
    if spark.catalog.tableExists(WATERMARK):
        res = spark.table(WATERMARK).filter("layer = 'silver'").select("last_processed_date").collect()
        if res and res[0][0]:
            return res[0][0]
    return None

# Update watermark after merge
def update_watermark(last_date):
    spark.createDataFrame([("silver", last_date, datetime.now(timezone.utc), BATCH_ID)],
                          "layer string, last_processed_date date, updated_at timestamp, last_batch_id string") \
         .createOrReplaceTempView("_wm")
    spark.sql(f"""
        MERGE INTO {WATERMARK} t USING _wm s ON t.layer = s.layer
        WHEN MATCHED THEN UPDATE SET t.last_processed_date = s.last_processed_date, t.updated_at = s.updated_at, t.last_batch_id = s.last_batch_id
        WHEN NOT MATCHED THEN INSERT *
    """)

# Audit execution tracker
class AuditLogger:
    def __init__(self, layer, param):
        self.layer, self.param = layer, param
        self.run_id = str(uuid.uuid4())
        self.read = self.ins = self.upd = self.quar = 0
    def __enter__(self):
        self.start = datetime.now(timezone.utc)
        return self
    def __exit__(self, exc_type, exc, tb):
        end = datetime.now(timezone.utc)
        status = "Failure" if exc_type else "Success"
        err = str(exc)[:2000] if exc_type else None
        spark.createDataFrame([(
            self.run_id, BATCH_ID, self.layer, "merge", str(self.param),
            self.start, end, (end - self.start).total_seconds(), status,
            int(self.read), int(self.ins), int(self.upd), int(self.quar), err
        )], schema="""run_id string, batch_id string, layer string, load_type string, parameter string, 
                     start_time timestamp, end_time timestamp, duration_seconds double, status string, 
                     rows_read long, rows_inserted long, rows_updated long, rows_quarantined long, error_message string""") \
        .write.format("delta").mode("append").saveAsTable(LOGS)
        return False

with AuditLogger("Bronze-to-Silver", f"batch={BATCH_FILTER}, start={START_DATE_FILTER}") as audit:
    bronze_df = spark.table(BRONZE)

    if BATCH_FILTER:
        bronze_df = bronze_df.filter(F.col("batch_id") == BATCH_FILTER)
    elif START_DATE_FILTER:
        bronze_df = bronze_df.filter(F.col("run_date") >= F.to_date(F.lit(START_DATE_FILTER)))
    else:
        wm = get_watermark()
        if wm:
            bronze_df = bronze_df.filter(F.col("run_date") > F.lit(wm))

    audit.read = bronze_df.count()

    if audit.read > 0:
        # Cast timestamps and determine invalid rows
        parsed_df = bronze_df \
            .withColumn("p_rec", parse_flexible_timestamp("date_received")) \
            .withColumn("p_sent", parse_flexible_timestamp("date_sent_to_company"))

        bad_condition = F.col("complaint_id").isNull() | (F.trim(F.col("complaint_id")) == "") | F.col("p_rec").isNull()

        # Route invalid records to quarantine
        bad_df = parsed_df.filter(bad_condition) \
            .withColumn("quarantine_reason", F.when(F.col("complaint_id").isNull(), "Missing ID").otherwise("Invalid Date")) \
            .withColumn("load_timestamp", F.current_timestamp())

        audit.quar = bad_df.count()
        if audit.quar > 0:
            bad_df.select(
                "product", "date_sent_to_company", "issue", "sub_product", "zip_code", "tags",
                "complaint_id", "timely", "company_response", "submitted_via", "company",
                "date_received", "state", "company_public_response", "sub_issue",
                "quarantine_reason", "source_file", "batch_id", "run_date", "load_timestamp"
            ).write.format("delta").mode("append").saveAsTable(QUARANTINE)

        # Standardize valid rows
        good_df = parsed_df.filter(~bad_condition).select(
            F.trim(F.col("complaint_id")).alias("complaint_id"),
            F.col("p_rec").alias("date_received"),
            F.col("p_sent").alias("date_sent_to_company"),
            F.trim(F.lower(F.col("product"))).alias("product"),
            F.trim(F.lower(F.col("sub_product"))).alias("sub_product"),
            F.trim(F.col("issue")).alias("issue"),
            F.trim(F.col("sub_issue")).alias("sub_issue"),
            F.trim(F.col("company")).alias("company"),
            F.upper(F.trim(F.col("state"))).alias("state"),
            F.trim(F.col("zip_code")).alias("zip_code"),
            F.trim(F.col("submitted_via")).alias("submitted_via"),
            F.trim(F.col("company_response")).alias("company_response"),
            F.trim(F.col("company_public_response")).alias("company_public_response"),
            (F.lower(F.trim(F.col("timely"))) == "yes").alias("timely"),
            F.col("tags"),
            F.current_timestamp().alias("load_timestamp"),
            F.lit(BATCH_ID).alias("batch_id")
        )

        # Deduplicate inside batch
        clean_df = good_df.withColumn("rn", F.expr("ROW_NUMBER() OVER (PARTITION BY complaint_id ORDER BY load_timestamp DESC)")) \
                          .filter("rn = 1").drop("rn")

        # Idempotent MERGE into Silver
        silver_delta = DeltaTable.forName(spark, SILVER)
        silver_delta.alias("t").merge(clean_df.alias("s"), "t.complaint_id = s.complaint_id") \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()

        # Capture metrics
        m = spark.sql(f"DESCRIBE HISTORY {SILVER} LIMIT 1").select("operationMetrics").first()[0]
        audit.ins = int(m.get("numTargetRowsInserted", 0))
        audit.upd = int(m.get("numTargetRowsUpdated", 0))

        # Advance watermark
        max_dt = clean_df.select(F.max(F.to_date("date_received"))).first()[0]
        if max_dt:
            update_watermark(max_dt)

print(f"Silver merge complete: {audit.ins} inserted, {audit.upd} updated, {audit.quar} quarantined.")


In [0]:
display(clean_df)